# Gold recovery in rougher flotation

Source from the original analysis. See the README for local inputs.


In [ ]:
import numpy as np 
import math
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance
from scipy.optimize import linprog

In [ ]:
from pathlib import Path

fileName = 'data/flotation.csv'
dateColumn = 'date'
df = pd.read_csv(fileName, index_col = dateColumn, parse_dates = [dateColumn])

## Data preprocessing


### Data inspection


In [ ]:
# check the overall info about the dataframe 
df.info()

In [ ]:
# check the percentage of missingness for each variable
print('Missing data fractions:')
for column_name in list(df.columns):
    missingpercentage = 100 * df[column_name].isnull().sum() / len(df)
    print(f'{missingpercentage:4.1f} % missing in {column_name}')

In [ ]:
# check how the missing values are distributed in the data frame 

fig, ax, = plt.subplots(figsize = (10, 10))
im = ax.pcolor(df.isnull())
ax.set_xlabel('Variable number')
ax.set_ylabel('Sample number')
cbar = plt.colorbar(im)
cbar.set_label('Missigness : 1 = missing, 0 = present')

In [ ]:
# check all the missing data and their timestamps 
all_missing = df.isnull().all(axis=1)

print("Rows with all measurements missing:", all_missing.sum())
print("Their timestamps:")
print(df.index[all_missing])

In [ ]:
# Checking for missing timestamps, dupilcates, and the order of the dataframe
print("Missing timestamps:", df.index.isnull().sum())
print("Repeated timestamps:", df.index.duplicated().sum())
print("Chronological order:", df.index.is_monotonic_increasing)

# Time differences between successive observations
time_steps = df.index.to_series().diff()

print("\nMost common time intervals:")
print(time_steps.value_counts().head(10))

In [ ]:
# Inspecting the whole data set
df.describe(include = 'all').T

In [ ]:
# Inspecting air flow to assess the negative readings 
air_columns = ['Floatbank10_a_air_[m3/h]','Floatbank10_b_air_[m3/h]', 'Floatbank10_c_air_[m3/h]', 'Floatbank10_d_air_[m3/h]', 'Floatbank10_e_air_[m3/h]', 'Floatbank10_f_air_[m3/h]']

df[air_columns].plot(subplots = True,  figsize = (12, 12))

In [ ]:
# Negative readings in the six air-flow variables

print("Negative air-flow counts:")
print((df[air_columns] < 0).sum())

In [ ]:
# Exact zero readings in the dataframe
zero_counts = (df == 0).sum()

print("\nZero-value counts:")
print(zero_counts[zero_counts > 0])

In [ ]:
# Assess where these happens in each of the variables

zero_columns = ['Feed_comp_ag[%]', 'Feed_comp_pb[%]', 'Feed_solids[%]','Feed_comp_au[%]','Concentrate_ag[%]', 'Concentrate_pb[%]','Concentrate_au[%]','Recovery[%]', 'Tailings_pb[%]']

df[zero_columns].plot(subplots = True, figsize = (12, 12))

In [ ]:
# check the box plot for each variables 
n_columns = 4
n_rows = math.ceil(len(df.columns) / n_columns)

fig, axes = plt.subplots(
    n_rows, n_columns,
    figsize=(16, 3 * n_rows)
)

axes = axes.flatten()

for ax, column in zip(axes, df.columns):
    ax.boxplot(df[column].dropna()) # do not include null values in the box-plot
    ax.set_title(column, fontsize=9)
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)

# Hide unused plotting areas
for ax in axes[len(df.columns):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

### Data cleaning


In [ ]:
# Preserve the original dataframe
df_copy = df

In [ ]:
# removing  recovery missed obeservations, since recovery is our prediction target and filling
# with estimated values can make the model learn for the inveted target 

df_copy.dropna(subset = 'Recovery[%]', inplace = True)

In [ ]:
# replace the negative air flow rates with NaN, supplied air flow cannot physically be negative 

for column in air_columns:
    df_copy[column] = df_copy[column].mask(
        df_copy[column] < 0)

In [ ]:
# flag observations outside grid-bearing slurry production 
invalid_feed = (
    (df_copy['Feed_comp_au[%]'] <= 0) | (df_copy['Feed_solids[%]'] <= 0) | (df_copy['Feed_rate_[ton/h]'] <= 0)
)

# filtering them out
df_copy = df_copy.loc[invalid_feed == False]

In [ ]:
# model inputs: feed conditions and operaring variables 

input_columns = [
    column for column in df_copy.columns
    if column.startswith('Feed')
    or column.startswith('Floatbank10_')
]

# Inspect remaining missing inputs 
missing_inputs = df_copy[input_columns].isnull().sum()

print('Missing measuremnents in proposed inputs: ')
print(missing_inputs[missing_inputs > 0])

In [ ]:
# Assess complete-case removal
complete_inputs = df_copy[input_columns].notna().all(axis = 1)

print('Current rows:', len(df_copy))
print('Rows with complete inputs:', complete_inputs.sum())
print('Rows with missing inputs:', (complete_inputs == False).sum())

In [ ]:
# check the pattern and distribution of these missing values 

fig, ax = plt.subplots(figsize = (14, 7))

heatmap = ax.imshow(
    df_copy[input_columns].isnull().to_numpy(),
    aspect = 'auto',
    interpolation = 'nearest',
    origin = 'upper',
    vmin = 0,
    vmax = 1
)

ax.set_xticks(range(len(input_columns)))
ax.set_xticklabels(input_columns, rotation = 90, fontsize = 8)

ax.set_xlabel('Model input')
ax.set_ylabel('Row position')
ax.set_title('Missing measurements in model inputs')

colourbar = fig.colorbar(heatmap, ax = ax, ticks = [0, 1])
colourbar.ax.set_yticklabels(['Present', 'Missing'])

plt.tight_layout()
plt.show()

In [ ]:
# Remove observation with missing feed size or feed solids (better option than inventing data for model inputs)

df_copy = df_copy.dropna(
    subset = ['Feed_size_[um]', 'Feed_solids[%]']
)

In [ ]:
# Identify inputs that still conatin missing measurements 

remaining_missing = df_copy[input_columns].isnull().sum()

for column in remaining_missing[remaining_missing > 0].index:
    print(f'{column}')
    print(df_copy.index[df_copy[column].isnull()])

In [ ]:
# Given how few missing observations remain, thus it is reasonable to remove them rather than
# making unsupported estimates 

df_copy = df_copy.dropna(subset = input_columns)

In [ ]:
# Compare dataset sizes 
print('Original row:', len(df))
print('Cleaned rows:', len(df_copy))
print('Rows removed:', len(df) - len(df_copy))
print(f'Data retained: {(len(df_copy)/len(df)) * 100:.2f}%')

In [ ]:
# confirm that model inputs have no missing values 

print('Remaining missing inputs: ')
print(df_copy[input_columns].isnull().sum())

In [ ]:
# confirm recovery has no missing values 
print('Reamining missing recovery:', df_copy['Recovery[%]'].isnull().sum())

In [ ]:
# Comapre summary statistics (for the original data set and cleaned data set)

comparison = pd.concat(
    [
        df[input_columns + ['Recovery[%]']].describe().T,
        df_copy[input_columns + ['Recovery[%]']].describe().T
    ],
    axis = 1,
    keys = ['Original', 'Cleaned']
)

comparison

## Model development and evaluation


In [ ]:
# Arrange observation chronologically

model_data = df_copy.sort_index()

X = model_data[input_columns]
y = model_data['Recovery[%]']

# Define split boudaries
train_end = int(0.70 * len(X))
validation_end = int(0.85 * len(X))

# Split by row position
X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:validation_end]
X_test = X.iloc[validation_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:validation_end]
y_test = y.iloc[validation_end:]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

for name, subset in [
    ('Training', X_train),
    ('Validation', X_val),
    ('Test', X_test)
]:
    print(
        f'{name}: {len(subset)} rows | '
        f'{subset.index.min()} to {subset.index.max()}'
    )

### Candidate models


In [ ]:
# Define the model (Linear regression)
linear_model = LinearRegression()

# Learn coefficients from trating inputs and measured recovery
linear_model.fit(X_train_scaled, y_train)

# Predict recovery for validation observation 
y_val_pred_linear = linear_model.predict(X_val_scaled)

# Define candidate models (k-nearest neighbors)
knn_models = {
    'KNN_5_uniform': KNeighborsRegressor(
        n_neighbors = 5, weights = 'uniform'
    ),
    'KNN_15_uniform': KNeighborsRegressor(
        n_neighbors = 15, weights = 'uniform'
    ),
    'KNN_30_uniform': KNeighborsRegressor(
        n_neighbors = 30, weights = 'uniform'
    ),
    'KNN_15_distance': KNeighborsRegressor(
        n_neighbors = 15, weights = 'distance'
    )
}

knn_val_predictions = {}

# Fit each candidate and predict validation recovery
for name, model in knn_models.items():
    model.fit(X_train_scaled, y_train)
    knn_val_predictions[name] = model.predict(X_val_scaled)


# Candidate hidden layers (neural_network)
nn_structures = {
    'NN_16': (16,),
    'NN_32': (32,),
    'NN_32_16': (32, 16)
}

nn_models = {}
nn_val_predictions = {}

for name, structure in nn_structures.items():
    model = MLPRegressor(
        hidden_layer_sizes = structure, 
        activation = 'relu', 
        max_iter = 5000,
        shuffle = False,
        random_state = 42 # for consistant results 
    )
    
    model.fit(X_train_scaled, y_train)
    
    nn_models[name] = model
    nn_val_predictions[name] = model.predict(X_val_scaled)

### Validation and model selection


In [ ]:
validation_predictions = {
    'Linear regression': y_val_pred_linear,
    **knn_val_predictions, 
    **nn_val_predictions
}

validation_results = pd.DataFrame([
    {'Model': name, 
    'Validation R^2': r2_score(y_val, predictions)} for name, predictions in validation_predictions.items()]).set_index('Model').sort_values('Validation R^2', ascending = False)

validation_results

In [ ]:
# model to advance - Linear regression 

fig, axes = plt.subplots(1, 2, figsize = (14, 5))

# validation recovery over time 
axes[0].plot(
    y_val.index, y_val,
    label = 'Measured'
)

axes[0].plot(
    y_val.index, y_val_pred_linear,
    label = 'Predicted'
)

axes[0].set_xlabel('Date')
axes[0].set_ylabel('Recovery[%]')
axes[0].set_title('Linear Regression: validation predictions')
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(True)
axes[0].legend()

# parity plot

axes[1].scatter(y_val, y_val_pred_linear, s = 12, alpha = 0.4)

lower = min(y_val.min(), y_val_pred_linear.min())
upper = max(y_val.max(), y_val_pred_linear.max())

axes[1].plot(
    [lower, upper], [lower, upper], 'k--', label = 'Perfect prediction'
)

axes[1].set_xlim(lower, upper)
axes[1].set_ylim(lower, upper)
axes[1].set_aspect('equal', adjustable = 'box')
axes[1].set_xlabel('Measured recovery [%]')
axes[1].set_ylabel('Predicted recovery [%]')
axes[1].set_title('Validation parity plot')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

### Test evaluation and permutation importance


In [ ]:
selected_model = linear_model 

y_test_pred = selected_model.predict(X_test_scaled)
test_r2 = r2_score(y_test, y_test_pred)

print(f'Validation R^2: {r2_score(y_val, y_val_pred_linear):.4f}')
print(f'Test R^2: {test_r2:4f}')

In [ ]:
# Selected linear model coefficients in both input scales
coefficient_table = pd.DataFrame({
    'Standardised coefficient': selected_model.coef_,
    'Original-unit coefficient': selected_model.coef_ / scaler.scale_
}, index=X_train.columns)

print(f'Standardised-input intercept: {selected_model.intercept_:.6f}')
original_intercept = selected_model.intercept_ - np.sum(selected_model.coef_ * scaler.mean_ / scaler.scale_)
print(f'Original-unit intercept: {original_intercept:.6f}')
display(coefficient_table)


In [ ]:
# inspect test predictions 
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

# Test recovery over time 
axes[0].plot(
    y_test.index, y_test,
    label = 'Measured'
)

axes[0].plot(
    y_test.index, y_test_pred,
    label = 'Predicted'
)

axes[0].set_xlabel('Date')
axes[0].set_ylabel('Recovery[%]')
axes[0].set_title('Linear Regression: test predictions')
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(True)
axes[0].legend()

# parity plot

axes[1].scatter(y_test, y_test_pred, s = 12, alpha = 0.4)

lower = min(y_test.min(), y_test_pred.min())
upper = max(y_test.max(), y_test_pred.max())

axes[1].plot(
    [lower, upper], [lower, upper], 'k--', label = 'Perfect prediction'
)

axes[1].set_xlim(lower, upper)
axes[1].set_ylim(lower, upper)
axes[1].set_aspect('equal', adjustable = 'box')
axes[1].set_xlabel('Measured recovery [%]')
axes[1].set_ylabel('Predicted recovery [%]')
axes[1].set_title(f'Test parity plot: R^2 = {test_r2:4f}')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
importance = permutation_importance(
    selected_model,
    X_test_scaled,
    y_test,
    n_repeats = 20,
    random_state = 42
)

# sort inputs by mean importance
order = np.argsort(importance.importances_mean)

means = importance.importances_mean[order]
stds = importance.importances_std[order]
names = X_train.columns.to_numpy()[order]

fig, ax = plt.subplots(figsize = (10, 8))

ax.barh(names, means, xerr = stds, capsize = 3, color = ['steelblue' if value >= 0 else 'darkorange' for value in means])

ax.axvline(0, color = 'black')
ax.set_xlabel('Mean decrease in test R^2 after suffling')
ax.set_ylabel('Model input')
ax.set_title('Permuation importance: selected linear model')
ax.grid(axis = 'x', alpha = 0.3)

plt.tight_layout()
plt.show()

## Operating-condition optimisation


In [ ]:
import json

fixed_feed = json.loads(Path('data/fixed_feed.json').read_text())

# Operating varibales to optimise 
Operating_columns = [column for column in X_train.columns if column.startswith('Floatbank10_')]

# Allowable range from the original training inputs 
Operating_bounds = pd.DataFrame({
    'Lower_bound': X_train[Operating_columns].min(),
    'Upper_bound': X_train[Operating_columns].max()
})

Operating_bounds

In [ ]:
# Convert fitted coefficients to original input units 
original_coefficients = pd.Series(selected_model.coef_ / scaler.scale_, index = X_train.columns)

# Coefficients for the operating varables 
Operating_coefficients = original_coefficients.loc[Operating_columns]

bounds = list(zip(
    Operating_bounds.loc[Operating_columns, 'Lower_bound'],
    Operating_bounds.loc[Operating_columns, 'Upper_bound']
))

# Maximise recovery by minising its negavtive contribution 
result = linprog(
    c = - Operating_coefficients.to_numpy(),
    bounds = bounds,
)

if not result.success:
    raise RuntimeError(result.message)

optimal_settings = pd.Series(result.x, index = Operating_columns, name = 'Optimal setting')

optimal_settings

In [ ]:
# Assemble all twenty inputs in the training column order 
optimal_input = X_train.iloc[[0]]

for column, value in fixed_feed.items():
    optimal_input[column] = value

optimal_input.loc[:, Operating_columns] = [optimal_settings.to_numpy()]

# Apply the existing training scaler and predict 
optimal_recovery = selected_model.predict(
    scaler.transform(optimal_input)
)[0]

# Define typical operation using training median settings 
typical_input = optimal_input

typical_input.loc[:, Operating_columns] = [X_train[Operating_columns].median().to_numpy()]

typical_recovery = selected_model.predict(
    scaler.transform(typical_input)
)[0]

# Display bounds, typical settings and optimum

settings_comparison = Operating_bounds
settings_comparison['Typical setting'] = (X_train[Operating_columns].median())
settings_comparison['Optimal setting'] = optimal_settings

display(settings_comparison)

print(f"Typical predicted recovery: {typical_recovery:.2f}%")
print(f"Optimal predicted recovery: {optimal_recovery:.2f}%")
print("Predicted improvement: ", f"{optimal_recovery - typical_recovery:.2f} percentage points")

In [ ]:
# assesing whether the specified feed is represented in training data 

feed_columns = list(fixed_feed)

feed_check = pd.DataFrame({
    'Specified feed': pd.Series(fixed_feed),
    'Training minimum': X_train[feed_columns].min(),
    "Training 25%": X_train[feed_columns].quantile(0.25),
    "Training median": X_train[feed_columns].median(),
    "Training 75%": X_train[feed_columns].quantile(0.75),
    "Training maximum": X_train[feed_columns].max()
})

feed_check['Within training range'] = ((feed_check['Specified feed'] >= feed_check['Training minimum']) & (feed_check['Specified feed'] <= feed_check['Training maximum']))

feed_check

In [ ]:
# Reuse the training scaler's parameters for the six feed inputs

feed_columns = list(fixed_feed)

feed_mean = pd.Series(scaler.mean_, index=X_train.columns).loc[feed_columns]

feed_scale = pd.Series(scaler.scale_, index=X_train.columns).loc[feed_columns]

training_feed_scaled = (X_train[feed_columns] - feed_mean) / feed_scale

specified_feed_scaled = (pd.Series(fixed_feed).loc[feed_columns] - feed_mean) / feed_scale

# Distance across all six feed measurements 
feed_distances = np.sqrt(((training_feed_scaled - specified_feed_scaled) ** 2).sum(axis=1))

nearest_dates = feed_distances.sort_values().head(5).index

nearest_feed = X_train.loc[nearest_dates, feed_columns]
nearest_feed["Standardised distance"] = feed_distances.loc[nearest_dates]

nearest_feed